In [ ]:
print("H")

In [ ]:
# Shared setup — imports, constants, paths
import os, json, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
from facenet_pytorch import MTCNN
from scipy.fft import dctn
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, roc_curve, precision_recall_curve,
)
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")

# Backend-compatible constants
KYC_MAX_VIDEO_FRAMES = 5
KYC_FRAME_SIZE       = 224

ROOT            = Path("../../")
DATA_DIR        = ROOT / "data" / "kyc"
CROPS_DIR       = DATA_DIR / "crops"
WEIGHTS_DIR     = ROOT / "backend" / "weights"
PRECOMPUTED_DIR = ROOT / "backend" / "precomputed"
RESULTS_DIR     = Path("results")
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
PRECOMPUTED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED   = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("Device:", DEVICE)

In [ ]:
# Build video-level 80/10/10 split from pre-extracted face crops
# Crop filenames: <source>_<videostem>_f<N>.jpg
# The split is at video level so no video leaks across splits.

def build_split(label):
    crops = list((CROPS_DIR / label).glob("*.jpg"))
    groups = {}
    for p in crops:
        key = p.stem.rsplit("_", 1)[0]
        groups.setdefault(key, []).append(p)
    videos = list(groups.keys())
    train_v, temp_v = train_test_split(videos, test_size=0.2, random_state=SEED)
    val_v,   test_v = train_test_split(temp_v, test_size=0.5, random_state=SEED)
    return {s: [p for v in vs for p in groups[v]]
            for s, vs in [("train",train_v),("val",val_v),("test",test_v)]}

real_splits = build_split("real")
fake_splits = build_split("fake")

split_dfs = {}
for split in ("train", "val", "test"):
    rows = [(str(p), 0) for p in real_splits[split]] + \
           [(str(p), 1) for p in fake_splits[split]]
    split_dfs[split] = pd.DataFrame(rows, columns=["path", "label"])
    n  = len(split_dfs[split])
    nr = (split_dfs[split].label == 0).sum()
    nf = (split_dfs[split].label == 1).sum()
    print(f"{split:5s}: {n} samples  (real={nr}, fake={nf})")

test_paths = split_dfs["test"]["path"].values

In [ ]:
# ImageNet normalization — same as backend inference.py
IMAGENET_TRANSFORM = transforms.Compose([
    transforms.Resize((KYC_FRAME_SIZE, KYC_FRAME_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])
AUGMENT_TRANSFORM = transforms.Compose([
    transforms.Resize((KYC_FRAME_SIZE, KYC_FRAME_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

class FaceDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform or IMAGENET_TRANSFORM
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return self.transform(Image.open(row["path"]).convert("RGB")), int(row["label"])

def make_loaders(batch_size=32):
    train_dl = DataLoader(FaceDataset(split_dfs["train"], AUGMENT_TRANSFORM),
                          batch_size=batch_size, shuffle=True,  num_workers=2)
    val_dl   = DataLoader(FaceDataset(split_dfs["val"]),
                          batch_size=batch_size, shuffle=False, num_workers=2)
    test_dl  = DataLoader(FaceDataset(split_dfs["test"]),
                          batch_size=batch_size, shuffle=False, num_workers=2)
    return train_dl, val_dl, test_dl

In [ ]:
# FrequencyCNN — matches backend/models/kyc/frequency_cnn.py exactly
class FrequencyCNN(nn.Module):
    def __init__(self, in_channels=1, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),          nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),         nn.ReLU(), nn.AdaptiveAvgPool2d(4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*4*4, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )
    def forward(self, x): return self.classifier(self.features(x))

In [ ]:
# DCT feature extractor — matches backend frequency_analyzer.extract_dct_features exactly
def extract_dct_features(face_image, size=224):
    gray    = face_image.convert("L").resize((size, size))
    arr     = np.array(gray, dtype=np.float32) / 255.0
    dct     = dctn(arr, norm="ortho")
    log_mag = np.log1p(np.abs(dct))
    log_mag = (log_mag - log_mag.min()) / (log_mag.max() - log_mag.min() + 1e-8)
    return log_mag[np.newaxis, :, :]   # (1, H, W)

class DCTDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        dct = extract_dct_features(Image.open(row["path"]).convert("RGB"))
        return torch.tensor(dct, dtype=torch.float), int(row["label"])

def make_dct_loaders(batch_size=32):
    train_dl = DataLoader(DCTDataset(split_dfs["train"]), batch_size=batch_size, shuffle=True,  num_workers=2)
    val_dl   = DataLoader(DCTDataset(split_dfs["val"]),   batch_size=batch_size, shuffle=False, num_workers=2)
    test_dl  = DataLoader(DCTDataset(split_dfs["test"]),  batch_size=batch_size, shuffle=False, num_workers=2)
    return train_dl, val_dl, test_dl

In [ ]:
# Training utilities shared by all RGB experiments

def train_one_epoch(model, loader, optimizer, criterion):
    model.train(); total_loss = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward(); optimizer.step()
        total_loss += loss.item() * len(y)
    return total_loss / len(loader.dataset)

def evaluate_loader(model, loader):
    model.eval(); all_labels, all_probs = [], []
    with torch.no_grad():
        for x, y in loader:
            prob = F.softmax(model(x.to(DEVICE)), dim=1)[:,1].cpu().numpy()
            all_probs.extend(prob); all_labels.extend(y.numpy())
    return np.array(all_labels), np.array(all_probs)

def train_model(model, train_dl, val_dl, optimizer, scheduler,
                epochs=20, save_path=None, patience=5):
    criterion = nn.CrossEntropyLoss()
    best_auc, wait = 0.0, 0
    history = {"train_loss": [], "val_auc": []}
    for epoch in range(1, epochs + 1):
        loss = train_one_epoch(model, train_dl, optimizer, criterion)
        labels, probs = evaluate_loader(model, val_dl)
        val_auc = roc_auc_score(labels, probs)
        history["train_loss"].append(loss)
        history["val_auc"].append(val_auc)
        if scheduler: scheduler.step()
        print(f"Epoch {epoch:02d}/{epochs}  loss={loss:.4f}  val_auc={val_auc:.4f}")
        if val_auc > best_auc:
            best_auc, wait = val_auc, 0
            if save_path:
                torch.save(model.state_dict(), save_path)
                print(f"  -> Checkpoint saved (auc={best_auc:.4f})")
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stopping at epoch {epoch}"); break
    return history

In [ ]:
# precompute.ipynb — Generate all backend artifacts from trained models
# Run this notebook after exp2.2, exp2.3, exp2.4 have completed.
# Outputs:
#   backend/weights/efficientnet_best.pt   (already saved by exp2.2)
#   backend/weights/vit_best.pt            (already saved by exp2.3)
#   backend/weights/frequency_cnn_best.pt  (already saved by exp2.4)
#   backend/precomputed/kyc_metrics.json   (generated here)

# Load all three production models
EFFNET_CKPT = WEIGHTS_DIR / "efficientnet_best.pt"
VIT_CKPT    = WEIGHTS_DIR / "vit_best.pt"
FREQ_CKPT   = WEIGHTS_DIR / "frequency_cnn_best.pt"

for ckpt in [EFFNET_CKPT, VIT_CKPT, FREQ_CKPT]:
    status = "OK" if ckpt.exists() else "MISSING"
    print(f"  {ckpt.name:32s}: {status}")

In [ ]:
# Load model weights
effnet = timm.create_model("efficientnet_b4", pretrained=False, num_classes=2)
effnet.load_state_dict(torch.load(EFFNET_CKPT, map_location=DEVICE))
effnet.eval().to(DEVICE)

vit = timm.create_model("vit_base_patch16_224", pretrained=False, num_classes=2)
vit.load_state_dict(torch.load(VIT_CKPT, map_location=DEVICE))
vit.eval().to(DEVICE)

freq_cnn = FrequencyCNN().to(DEVICE)
freq_cnn.load_state_dict(torch.load(FREQ_CKPT, map_location=DEVICE))
freq_cnn.eval()

print("All models loaded.")

In [ ]:
# Run full test-set inference for all three models
train_dl, val_dl, test_dl = make_loaders(batch_size=32)
dct_test_dl = DataLoader(
    __import__("torch").utils.data.Dataset.__class__,   # placeholder
    batch_size=32, shuffle=False
)

# RGB loaders
labels_eff, probs_eff = evaluate_loader(effnet,   test_dl)
labels_vit, probs_vit = evaluate_loader(vit,       test_dl)

# DCT loader for FrequencyCNN
class DCTDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        dct = extract_dct_features(Image.open(row["path"]).convert("RGB"))
        return torch.tensor(dct, dtype=torch.float), int(row["label"])

dct_test_dl = DataLoader(DCTDataset(split_dfs["test"]), batch_size=32,
                         shuffle=False, num_workers=2)
labels_freq, probs_freq = evaluate_loader(freq_cnn, dct_test_dl)

probs_ensemble = (probs_eff + probs_vit + probs_freq) / 3
print("Inference done.")

In [ ]:
# Compute metrics for each model
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, roc_curve
)

def get_metrics(y_true, probs, threshold=0.5):
    yp = (probs >= threshold).astype(int)
    return {
        "Accuracy":  round(accuracy_score(y_true, yp), 4),
        "Precision": round(precision_score(y_true, yp, zero_division=0), 4),
        "Recall":    round(recall_score(y_true, yp, zero_division=0), 4),
        "F1":        round(f1_score(y_true, yp, zero_division=0), 4),
        "ROC-AUC":   round(roc_auc_score(y_true, probs), 4),
        "PR-AUC":    round(average_precision_score(y_true, probs), 4),
    }

metrics_eff  = get_metrics(labels_eff,  probs_eff)
metrics_vit  = get_metrics(labels_vit,  probs_vit)
metrics_freq = get_metrics(labels_freq, probs_freq)
metrics_ens  = get_metrics(labels_eff,  probs_ensemble)

for name, m in [("EfficientNet-B4",metrics_eff),("ViT-B/16",metrics_vit),
                 ("FrequencyCNN",metrics_freq),("Ensemble",metrics_ens)]:
    print(f"{name}: F1={m['F1']} ROC-AUC={m['ROC-AUC']} PR-AUC={m['PR-AUC']}")

In [ ]:
# Build ROC curve data (downsampled to 100 pts for JSON size)
def roc_to_list(y_true, probs, n=100):
    fpr, tpr, thr = roc_curve(y_true, probs)
    idx = np.linspace(0, len(fpr)-1, min(n, len(fpr)), dtype=int)
    return [{"fpr":round(float(fpr[i]),4),"tpr":round(float(tpr[i]),4),
             "threshold":round(float(thr[i]),4)} for i in idx]

# Load robustness data from exp2.6 if it exists
rob_path = RESULTS_DIR / "exp2.6_robustness.csv"
robustness_records = pd.read_csv(rob_path).to_dict(orient="records") \
    if rob_path.exists() else []

# Assemble kyc_metrics.json
kyc_metrics = {
    "models": {
        "efficientnet_b4": {**metrics_eff,  "roc_curve": roc_to_list(labels_eff,  probs_eff)},
        "vit_b16":         {**metrics_vit,  "roc_curve": roc_to_list(labels_vit,  probs_vit)},
        "frequency_cnn":   {**metrics_freq, "roc_curve": roc_to_list(labels_freq, probs_freq)},
        "ensemble":        {**metrics_ens,  "roc_curve": roc_to_list(labels_eff,  probs_ensemble)},
    },
    "datasets": [
        {"name":"FaceForensics++","splits":{"train":0.8,"val":0.1,"test":0.1}},
        {"name":"Celeb-DF v2",    "splits":{"train":0.8,"val":0.1,"test":0.1}},
    ],
    "production_thresholds": {
        "verified":   {"lt": 0.50},
        "suspicious": {"gte": 0.50, "lt": 0.75},
        "high_risk":  {"gte": 0.75},
    },
    "preprocessing": {
        "max_frames":         KYC_MAX_VIDEO_FRAMES,
        "frame_size":         KYC_FRAME_SIZE,
        "face_detector":      "MTCNN",
        "rgb_normalization":  "ImageNet",
        "frequency_pipeline": "grayscale -> 2D-DCT -> log1p(abs) -> [0,1]",
    },
    "robustness": robustness_records,
}

out_path = PRECOMPUTED_DIR / "kyc_metrics.json"
with open(out_path, "w") as f:
    json.dump(kyc_metrics, f, indent=2)
print(f"Saved: {out_path}")

In [ ]:
# Backend compatibility check
dummy = Image.fromarray(np.random.randint(0,255,(224,224,3),dtype=np.uint8))

IMAGENET_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])
rgb_t = IMAGENET_TRANSFORM(dummy).unsqueeze(0).to(DEVICE)
assert rgb_t.shape == (1,3,224,224)

dct_arr = extract_dct_features(dummy)
assert dct_arr.shape == (1,224,224)
dct_t = torch.tensor(dct_arr,dtype=torch.float).unsqueeze(0).to(DEVICE)
assert dct_t.shape == (1,1,224,224)

with torch.no_grad():
    p_e = float(F.softmax(effnet(rgb_t),   dim=1)[0,1])
    p_v = float(F.softmax(vit(rgb_t),      dim=1)[0,1])
    p_f = float(F.softmax(freq_cnn(dct_t), dim=1)[0,1])

ens = (p_e + p_v + p_f) / 3
print(f"EfficientNet : {p_e:.4f}")
print(f"ViT          : {p_v:.4f}")
print(f"FrequencyCNN : {p_f:.4f}")
print(f"Ensemble     : {ens:.4f}")
print("All compatibility checks passed.")

In [ ]:
# Final artifact summary
print("\n=== Artifact Summary ===")
artifacts = [
    WEIGHTS_DIR / "efficientnet_best.pt",
    WEIGHTS_DIR / "vit_best.pt",
    WEIGHTS_DIR / "frequency_cnn_best.pt",
    PRECOMPUTED_DIR / "kyc_metrics.json",
]
for p in artifacts:
    size = f"{p.stat().st_size/1024:.0f} KB" if p.exists() else "MISSING"
    print(f"  {p.name:35s}: {size}")
print("precompute done")